# Pix2Pix — entrenamiento y métricas de pérdidas (versión existente)

Este notebook utiliza **la implementación existente** de `src/gan/losses.py` en la rama `feat/discriminator-losses`:

- `adversarial_loss`: BCE con logits.
- `l1_loss`: error absoluto medio.
- `discriminator_loss`: promedio de las pérdidas real/falsa.
- `generator_loss`: pérdida adversarial + `lambda_l1 * L1`.

No incorpora MSE/LSGAN, label smoothing, recorte de logits, R1 ni las funciones ampliadas que se compartieron después.

Para poder ejecutar el proyecto completo, toma la arquitectura U-Net y el entrenamiento que ya están en `main`, y sustituye **únicamente** `src/gan/losses.py` por la versión existente de `feat/discriminator-losses`. No modifica GitHub; el cambio ocurre solo en el entorno temporal de Colab.


## 1. Comprobar GPU

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU disponibles:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Descargar el repositorio y recuperar la versión existente de losses.py

Se clona `main` para obtener el generador y el bucle de entrenamiento ya integrados. Después se recupera **solo** el archivo `losses.py` de la rama de trabajo, sin hacer merge ni alterar el repositorio remoto.


In [ ]:
%cd /content
!git clone https://github.com/Isabela1105/Trabajo-Final-de-Redes.git
%cd /content/Trabajo-Final-de-Redes
!git fetch origin feat/discriminator-losses
!git show FETCH_HEAD:src/gan/losses.py > src/gan/losses.py
!git status --short


/content
fatal: destination path 'Trabajo-Final-de-Redes' already exists and is not an empty directory.
/content/Trabajo-Final-de-Redes
From https://github.com/Isabela1105/Trabajo-Final-de-Redes
 * branch            feat/discriminator-losses -> FETCH_HEAD
 M src/gan/losses.py


## 3. Instalar dependencias del notebook

Colab normalmente ya incluye TensorFlow. Se instalan las dependencias auxiliares utilizadas por la configuración, el dataset y las gráficas.


In [ ]:
!pip -q install pyyaml pillow pandas matplotlib


## 4. Conectar Google Drive y definir las carpetas del dataset

El cargador existente espera esta estructura:

```text
dataset/
  train/
    input/
    target/
  val/
    input/
    target/
  test/
    input/
    target/
```

Cada imagen de `input` debe tener su pareja con el mismo nombre en `target`. Las imágenes se cargan a 256×256 RGB y se normalizan a [-1, 1] por el módulo del proyecto.

Cambia `DATASET_ROOT` para que apunte a la carpeta real de tu Drive.


In [ ]:
from pathlib import Path

print("Archivos en /content:")
for archivo in Path("/content").iterdir():
    print(archivo.name)

Archivos en /content:
.config
Trabajo-Final-de-Redes
dataset
datasetv1(1).zip
sample_data


In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/datasetv1(1).zip")
EXTRACT_DIR = Path("/content/dataset")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extraído correctamente.")
print("\nCarpetas encontradas:")

for path in sorted(EXTRACT_DIR.rglob("*")):
    if path.is_dir():
        print(path)

Dataset extraído correctamente.

Carpetas encontradas:
/content/dataset/processed
/content/dataset/processed/test
/content/dataset/processed/test/input
/content/dataset/processed/test/target
/content/dataset/processed/train
/content/dataset/processed/train/input
/content/dataset/processed/train/target
/content/dataset/processed/val
/content/dataset/processed/val/input
/content/dataset/processed/val/target


## 5. Preparar configuración de entrenamiento

Se parte de `configs/base.yaml` del proyecto y se modifican únicamente las rutas de datos, carpeta de salida y cantidad de épocas para esta ejecución. Los hiperparámetros base se mantienen salvo que cambies explícitamente los valores aquí.


In [ ]:
from pathlib import Path
import yaml

# Definir las rutas del dataset que faltaban
TRAIN_DIR = Path("/content/dataset/processed/train")
VAL_DIR = Path("/content/dataset/processed/val")
TEST_DIR = Path("/content/dataset/processed/test")

with open("configs/base.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["train_dir"] = str(TRAIN_DIR)
cfg["data"]["val_dir"] = str(VAL_DIR)
if TEST_DIR.exists():
    cfg["data"]["test_dir"] = str(TEST_DIR)

# Ajustes de ejecución en Colab
cfg["train"]["epochs"] = 20
cfg["output"]["dir"] = "/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente"

# Conservamos la implementación real; no permitir fallback de smoke.
cfg["train"]["allow_smoke_fallback"] = False

CONFIG_PATH = Path("/content/Trabajo-Final-de-Redes/configs/colab_metrics.yaml")
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print(CONFIG_PATH.read_text())

seed: 42
data:
  train_dir: /content/dataset/processed/train
  val_dir: /content/dataset/processed/val
  test_dir: /content/dataset/processed/test
  image_size: 256
  batch_size: 2
  num_parallel_calls: 4
model:
  ngf: 64
  ndf: 64
  n_downsample: 8
  dropout: 0.5
  norm: instance
  generator_builder: src.gan.models.generator:build_generator
  discriminator_builder: src.gan.models.discriminator:build_discriminator
  losses_module: src.gan.losses
train:
  allow_smoke_fallback: false
  max_to_keep: 3
  epochs: 20
  lr_g: 0.0002
  lr_d: 0.0002
  beta_1: 0.5
  beta_2: 0.999
  lambda_l1: 100.0
  label_smoothing: 0.0
  mixed_precision: true
  clip_norm: 1.0
  sample_interval: 5
  checkpoint_interval: 1
  early_stopping_patience: 15
  max_steps: null
output:
  dir: /content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente
  save_samples: true



## 6. Verificar que se importan los componentes reales

Esta celda confirma las firmas de las funciones de pérdida existentes y las dimensiones esperadas del discriminador PatchGAN.


In [ ]:
import inspect
from src.gan.losses import (
    adversarial_loss,
    l1_loss,
    discriminator_loss,
    generator_loss,
)
from src.gan.models.generator import build_generator
from src.gan.models.discriminator import build_discriminator

print("generator_loss:", inspect.signature(generator_loss))
print("discriminator_loss:", inspect.signature(discriminator_loss))
print("adversarial_loss:", inspect.signature(adversarial_loss))

G = build_generator(image_size=256, ngf=64, niveles=8, norm="instance", dropout=0.5)
D = build_discriminator(image_size=256, ndf=64)

dummy = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
fake = G(dummy, training=False)
logits = D([dummy, fake], training=False)

print("Entrada G:", dummy.shape)
print("Salida G:", fake.shape)
print("Salida D:", logits.shape)
assert tuple(fake.shape[1:]) == (256, 256, 3)
assert tuple(logits.shape[1:]) == (30, 30, 1)


generator_loss: (fake_logits, gen_output, target, lambda_l1=100.0)
discriminator_loss: (real_logits, fake_logits)
adversarial_loss: (logits, target_is_real)
Entrada G: (1, 256, 256, 3)
Salida G: (1, 256, 256, 3)
Salida D: (1, 30, 30, 1)


## 7. Ejecutar entrenamiento y registrar métricas

El bucle existente `src.gan.train` ya calcula y guarda `metrics.csv`, checkpoints, muestras y eventos de TensorBoard. No se vuelve a implementar ni se reemplaza la función de pérdida.


In [ ]:
from pathlib import Path

BASE = Path("/content/dataset")

print("Carpetas:")
for p in sorted(BASE.rglob("*")):
    if p.is_dir():
        print(p)

print("\nArchivos PNG encontrados:")
for p in list(BASE.rglob("*.png"))[:20]:
    print(p)

Carpetas:
/content/dataset/processed
/content/dataset/processed/test
/content/dataset/processed/test/input
/content/dataset/processed/test/target
/content/dataset/processed/train
/content/dataset/processed/train/input
/content/dataset/processed/train/target
/content/dataset/processed/val
/content/dataset/processed/val/input
/content/dataset/processed/val/target

Archivos PNG encontrados:
/content/dataset/processed/test/input/0898_v1.png
/content/dataset/processed/test/input/0898_v2.png
/content/dataset/processed/test/input/0820_v0.png
/content/dataset/processed/test/input/foto_040_v2.png
/content/dataset/processed/test/input/foto_011_v1.png
/content/dataset/processed/test/input/foto_014_v0.png
/content/dataset/processed/test/input/0848_v1.png
/content/dataset/processed/test/input/foto_003_v2.png
/content/dataset/processed/test/input/0842_v2.png
/content/dataset/processed/test/input/0848_v0.png
/content/dataset/processed/test/input/0837_v1.png
/content/dataset/processed/test/input/0848_

In [ ]:
print("CONFIG_PATH:", CONFIG_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    print(f.read())

CONFIG_PATH: /content/Trabajo-Final-de-Redes/configs/colab_metrics.yaml
seed: 42
data:
  train_dir: /content/dataset/processed/train
  val_dir: /content/dataset/processed/val
  test_dir: /content/dataset/processed/test
  image_size: 256
  batch_size: 2
  num_parallel_calls: 4
model:
  ngf: 64
  ndf: 64
  n_downsample: 8
  dropout: 0.5
  norm: instance
  generator_builder: src.gan.models.generator:build_generator
  discriminator_builder: src.gan.models.discriminator:build_discriminator
  losses_module: src.gan.losses
train:
  allow_smoke_fallback: false
  max_to_keep: 3
  epochs: 20
  lr_g: 0.0002
  lr_d: 0.0002
  beta_1: 0.5
  beta_2: 0.999
  lambda_l1: 100.0
  label_smoothing: 0.0
  mixed_precision: true
  clip_norm: 1.0
  sample_interval: 5
  checkpoint_interval: 1
  early_stopping_patience: 15
  max_steps: null
output:
  dir: /content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente
  save_samples: true



In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("/content/Trabajo-Final-de-Redes/configs/colab_metrics.yaml")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config_yaml = yaml.safe_load(f)

config_yaml["data"]["train_dir"] = "/content/dataset/processed/train"
config_yaml["data"]["val_dir"] = "/content/dataset/processed/val"
config_yaml["data"]["test_dir"] = "/content/dataset/processed/test"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config_yaml, f, sort_keys=False, allow_unicode=True)

print("Rutas actualizadas:")
print("Train:", config_yaml["data"]["train_dir"])
print("Val:", config_yaml["data"]["val_dir"])
print("Test:", config_yaml["data"]["test_dir"])

Rutas actualizadas:
Train: /content/dataset/processed/train
Val: /content/dataset/processed/val
Test: /content/dataset/processed/test


In [ ]:
from pathlib import Path

for split in ["train", "val", "test"]:
    for tipo in ["input", "target"]:
        carpeta = Path("/content/dataset/processed") / split / tipo
        imagenes = list(carpeta.glob("*.png"))

        print(f"{split}/{tipo}: {len(imagenes)} imágenes")
        assert len(imagenes) > 0, f"No hay imágenes en {carpeta}"

print("\nTodas las carpetas tienen imágenes.")

train/input: 366 imágenes
train/target: 366 imágenes
val/input: 45 imágenes
val/target: 45 imágenes
test/input: 45 imágenes
test/target: 45 imágenes

Todas las carpetas tienen imágenes.


In [ ]:
from src.gan.config import load_config

config = load_config(CONFIG_PATH)

print("Configuración cargada correctamente.")
print(type(config))
print(vars(config))

Configuración cargada correctamente.
<class 'src.gan.config.Config'>
{'seed': 42, 'data': {'train_dir': PosixPath('/content/dataset/processed/train'), 'val_dir': PosixPath('/content/dataset/processed/val'), 'test_dir': PosixPath('/content/dataset/processed/test'), 'image_size': 256, 'batch_size': 2, 'num_parallel_calls': 4}, 'model': {'ngf': 64, 'ndf': 64, 'n_downsample': 8, 'dropout': 0.5, 'norm': 'instance', 'generator_builder': 'src.gan.models.generator:build_generator', 'discriminator_builder': 'src.gan.models.discriminator:build_discriminator', 'losses_module': 'src.gan.losses'}, 'train': {'allow_smoke_fallback': False, 'max_to_keep': 3, 'epochs': 20, 'lr_g': 0.0002, 'lr_d': 0.0002, 'beta_1': 0.5, 'beta_2': 0.999, 'lambda_l1': 100.0, 'label_smoothing': 0.0, 'mixed_precision': True, 'clip_norm': 1.0, 'sample_interval': 5, 'checkpoint_interval': 1, 'early_stopping_patience': 15, 'max_steps': None}, 'output': {'dir': '/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente', '

In [ ]:
print(config.data)

{'train_dir': PosixPath('/content/dataset/processed/train'), 'val_dir': PosixPath('/content/dataset/processed/val'), 'test_dir': PosixPath('/content/dataset/processed/test'), 'image_size': 256, 'batch_size': 2, 'num_parallel_calls': 4}


In [ ]:
from src.gan.config import load_config

config = load_config(CONFIG_PATH)

print("Configuración cargada correctamente.")
print(config.data)

Configuración cargada correctamente.
{'train_dir': PosixPath('/content/dataset/processed/train'), 'val_dir': PosixPath('/content/dataset/processed/val'), 'test_dir': PosixPath('/content/dataset/processed/test'), 'image_size': 256, 'batch_size': 2, 'num_parallel_calls': 4}


In [ ]:
from src.gan.config import load_config
from src.gan.train import train

config = load_config(CONFIG_PATH)
metrics_path = train(config)

print("Archivo de métricas:", metrics_path)


[train] Mixed precision activada (pérdidas en float32).
época 000 | D 0.7124 | G 23.8054 | L1 val 0.1973
época 001 | D 0.2230 | G 18.3720 | L1 val 0.1745
época 002 | D 0.0116 | G 17.3904 | L1 val 0.1703
época 003 | D 0.0237 | G 16.7996 | L1 val 0.1683
época 004 | D 0.0083 | G 16.1700 | L1 val 0.1581
época 005 | D 0.0069 | G 15.6110 | L1 val 0.1605
época 006 | D 0.0016 | G 14.9292 | L1 val 0.1722
época 007 | D 0.0204 | G 15.4600 | L1 val 0.1591
época 008 | D 0.0305 | G 14.5448 | L1 val 0.1591
época 009 | D 0.0011 | G 13.9324 | L1 val 0.1656
época 010 | D 0.0395 | G 14.4714 | L1 val 0.1510
época 011 | D 0.0069 | G 13.9824 | L1 val 0.1540
época 012 | D 0.0115 | G 13.8062 | L1 val 0.1539
época 013 | D 0.0137 | G 13.8515 | L1 val 0.1568
época 014 | D 0.0080 | G 13.2989 | L1 val 0.1493
época 015 | D 0.0005 | G 13.4928 | L1 val 0.1580
época 016 | D 0.0005 | G 12.9443 | L1 val 0.1443
época 017 | D 0.0001 | G 12.4506 | L1 val 0.1469
época 018 | D 0.0966 | G 14.4826 | L1 val 0.1471
época 019 | D

## 8. Leer y visualizar las métricas de pérdida

El CSV del proyecto guarda filas separadas para `train` y `val`. Las columnas disponibles son `epoch`, `split`, `loss_g`, `loss_d`, `l1`, `psnr` y `ssim`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(metrics_path)
display(metrics.head(10))

train_metrics = metrics[metrics["split"] == "train"].copy()
val_metrics = metrics[metrics["split"] == "val"].copy()

for col in ("loss_g", "loss_d", "l1", "psnr", "ssim"):
    if col in metrics.columns:
        metrics[col] = pd.to_numeric(metrics[col], errors="coerce")

train_metrics = metrics[metrics["split"] == "train"]
val_metrics = metrics[metrics["split"] == "val"]

plt.figure(figsize=(10, 5))
plt.plot(train_metrics["epoch"], train_metrics["loss_g"], label="Generator loss (train)")
plt.plot(train_metrics["epoch"], train_metrics["loss_d"], label="Discriminator loss (train)")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Pérdidas del generador y discriminador")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(train_metrics["epoch"], train_metrics["l1"], label="L1 train")
plt.plot(val_metrics["epoch"], val_metrics["l1"], label="L1 validation")
plt.xlabel("Época")
plt.ylabel("L1")
plt.title("Error L1: entrenamiento vs validación")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(val_metrics["epoch"], val_metrics["psnr"], label="PSNR validation")
plt.xlabel("Época")
plt.ylabel("PSNR (dB)")
plt.title("PSNR en validación")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(val_metrics["epoch"], val_metrics["ssim"], label="SSIM validation")
plt.xlabel("Época")
plt.ylabel("SSIM")
plt.title("SSIM en validación")
plt.grid(True)
plt.legend()
plt.show()


,epoch,split,loss_g,loss_d,l1,psnr,ssim
0,0,train,23.805373,0.712361,0.231242,NaN,NaN
1,0,val,NaN,NaN,0.197257,18.151883,0.433984
2,1,train,18.372014,0.223013,0.178130,NaN,NaN
3,1,val,NaN,NaN,0.174468,19.187116,0.476413
4,2,train,17.390400,0.011604,0.168005,NaN,NaN
5,2,val,NaN,NaN,0.170283,19.404162,0.484066
6,3,train,16.799563,0.023740,0.159811,NaN,NaN
7,3,val,NaN,NaN,0.168284,19.463826,0.483303
8,4,train,16.170031,0.008255,0.155651,NaN,NaN
9,4,val,NaN,NaN,0.158062,19.710633,0.497395


## 9. Resumen numérico de la última época

Las pérdidas no tienen un umbral universal de “bueno” o “malo”; deben interpretarse junto con las curvas, las imágenes generadas y las métricas de validación.


In [ ]:
last_train = train_metrics.sort_values("epoch").tail(1)
last_val = val_metrics.sort_values("epoch").tail(1)

print("Última época de entrenamiento:")
display(last_train[["epoch", "loss_g", "loss_d", "l1"]])

print("Última época de validación:")
display(last_val[["epoch", "l1", "psnr", "ssim"]])


Última época de entrenamiento:


,epoch,loss_g,loss_d,l1
38,19,12.833614,0.000049,0.12443


Última época de validación:


,epoch,l1,psnr,ssim
39,19,0.148207,20.261772,0.507655


## 10. Descargar resultados

Los resultados quedan guardados en Google Drive en `pix2pix_outputs/metricas_loss_existente`, incluyendo `metrics.csv`, checkpoints, logs y muestras generadas.


In [ ]:
OUTPUT_DIR = Path(cfg["output"]["dir"])
print("Contenido de salida:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(p)


Contenido de salida:
/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/checkpoints
/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/config.yaml
/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/logs
/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/metrics.csv
/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/samples
